In [1]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re

/mnt/shared/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
COLLECTION_NAME = "freecad_docs"
DATA_PATH = "data"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "embeddings"

In [3]:
device = "cuda"

In [4]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

In [5]:
client = QdrantClient(path="qdrant")

In [6]:
def filter_code(doc: str) -> str:
    code_blocks = re.findall(r"```(?:[a-zA-Z]+\n)?(.*?)```", doc, re.DOTALL)
    code = "\n\n".join(code_blocks)
    return re.sub(r'\n{3,}', '\n\n', code)

In [7]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [8]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "tooltips_dataset.csv"))

In [11]:
query_idx = 0

query = query_df.iloc[query_idx]["Y"]
query_qdrant = query + "python code"
# query_qdrant = "Make a circle"


query_vec = embedding_model.encode([query_qdrant])[0]
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)

print(hits.points[0].payload)

retrieved_text = [point.payload['content'] for point in hits.points]
retrieved_code = [filter_code(text) for text in retrieved_text]

retrieved_vec = embedding_model.encode(retrieved_code)

y_vec = embedding_model.encode([query_df.iloc[query_idx]["Y"]])[0]

print(f"X: {query_df.iloc[query_idx]['X']}")
print("Y\n", query_df.iloc[query_idx]["Y"], "\n")
for i, vec in enumerate(retrieved_vec):
    print(f"Hit {i+1}, score: {cosine_similarity(vec, y_vec)}")

for i, text in enumerate(retrieved_text):
    print(f"=================Hit {i+1}=================")
    print(text)


{'source': 'Arch_Wall.wikitext', 'chunk_idx': 6, 'content': 'Scripting\n\n Arch API and FreeCAD Scripting Basics.\n\nThe Wall tool can be used in macros and from the Python console by using the following function:\n\n```start_code\n\nWall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")\n\nend_code```\n\n Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.\n If no  is given, you can provide the numerical values for the ,  (thickness), and .\n If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.\n  can be ,  or .\n It returns  if the operation fails.\n\nExample:\n\n```start_code\n\nimport FreeCAD, Draft, Arch\n\np1 = FreeCAD.Vector(0, 0, 0)\np2 = FreeCAD.Vector(2000, 0, 0)\nbaseline = Draft.makeLine(p1, p2)\nWall1 = Arch.makeWall(baseline, length=None, width=150, height=2000)\nFreeCAD.ActiveDocument.recompute(

In [9]:
doc = "Create a wall using python code"
query_vec = embedding_model.encode([doc])[0]
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)
for point in hits.points:
    print("===================\n")
    print("Score: ", point.score)
    print("Source", point.payload['source'])
    print(point.payload['content'])


Score:  0.6672807071944913
Source Arch_Wall.wikitext
Scripting

 Arch API and FreeCAD Scripting Basics.

The Wall tool can be used in macros and from the Python console by using the following function:

```

Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")

```

 Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.
 If no  is given, you can provide the numerical values for the ,  (thickness), and .
 If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.
  can be ,  or .
 It returns  if the operation fails.

Example:

```

import FreeCAD, Draft, Arch

p1 = FreeCAD.Vector(0, 0, 0)
p2 = FreeCAD.Vector(2000, 0, 0)
baseline = Draft.makeLine(p1, p2)
Wall1 = Arch.makeWall(baseline, length=None, width=150, height=2000)
FreeCAD.ActiveDocument.recompute()
```

Score:  0.6237083836818383
Source Arch_CurtainWall.wikit

In [10]:
for point in hits.points:
    print("===================\n")
    print("Source", point.payload['source'])
    print(filter_code(point.payload["content"]))


Source Arch_Wall.wikitext


Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")





import FreeCAD, Draft, Arch

p1 = FreeCAD.Vector(0, 0, 0)
p2 = FreeCAD.Vector(2000, 0, 0)
baseline = Draft.makeLine(p1, p2)
Wall1 = Arch.makeWall(baseline, length=None, width=150, height=2000)
FreeCAD.ActiveDocument.recompute()


Source Arch_CurtainWall.wikitext


MyCurtainWall = makeCurtainWall(baseobj)





import FreeCAD, Draft, Arch

p1 = FreeCAD.Vector(0, 0, 0)
p2 = FreeCAD.Vector(2000, 0, 0)
baseline = Draft.makeLine(p1, p2)
baseface = FreeCAD.ActiveDocument.addObject('Part::Extrusion','Extrusion')
baseface.Base = baseline
baseface.DirMode = "Normal"
baseface.LengthFwd = 2000
curtainwall = Arch.makeCurtainWall(baseface)
curtainwall.VerticalSections = 6
FreeCAD.ActiveDocument.recompute()



Source Arch_Floor.wikitext

Wall1 = Arch.makeWall(baseline, length=None, width=150, height=2000)
Wall2 = Arch.makeWall(baseline2, length=None, width=150,

In [23]:
responses = []
for doc in query_df['X']:
    query_vec = embedding_model.encode([doc])[0]
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=5,
    )
    top_results = []
    for point in hits.points:
        top_results.append(filter_code(point.payload["content"]))
    responses.append(top_results)
query_df["retrieved"] = responses

In [28]:
query_df

,X,Y,retrieved
0,Create a wall,import Arch\nobj = Arch.makeWall(),"[, , , , \n\nWall = makeWall(baseobj=None, len..."
1,Create a structural element,import Arch\nobj = Arch.makeStructure(length=1...,"[, , , , ]"
2,Add a window,"import Draft, Arch\nbase = Draft.makeRectangle...","[, , \n\nNote: You can even add other groups t..."
3,Add a door,"import Draft, Arch\nbase = Draft.makeRectangle...","[, \n\naddComponents(objectsList, host)\n\n, ,..."
4,Create a roof,"import Draft, Arch\nbase = Draft.makeRectangle...","[, \n\nRoof = makeRoof(baseobj=None, facenr=0,..."
...,...,...,...
245,Add parallelism (Assembly2+),FreeCADGui.runCommand(‘A2p_ConstraintParallel’),[\n\nfrom SlabReinforcement.SlabReinforcement ...
246,Add angular constraint (Assembly2+),FreeCADGui.runCommand(‘A2p_ConstraintAngle’),"[, , , , ]"
247,Add concentricity (Assembly2+),FreeCADGui.runCommand(‘A2p_ConstraintConcentric’),"[, , , \n\nAdd a volume with the element numbe..."
248,Insert part into assembly (Assembly4),FreeCADGui.runCommand(‘Asm4_InsertPart’),"[, \n\n if ob.Type[:4] == 'Part':\n ..."
